In [ ]:
# =======================
# Cell 1: Cài đặt thư viện
# =======================

%pip install -q "transformers>=4.45.0" "peft>=0.11.0" "accelerate" "qwen-vl-utils" "datasets" "tqdm"


In [2]:
# =======================
# Cell 2: Import & cấu hình chung
# =======================

import os
import json
from pathlib import Path
from typing import List, Dict, Any

import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import AdamW
from tqdm.auto import tqdm
from PIL import Image

from transformers import (
    Qwen2VLForConditionalGeneration,
    AutoProcessor,
    CLIPModel,
    CLIPProcessor,
)
from peft import LoraConfig, get_peft_model

from qwen_vl_utils import process_vision_info  # dùng cho Qwen2-VL

# ==== Đường dẫn ====
DATA_ROOT = Path("/kaggle/input/datasetrm2")

# JSONL_PATH = Path("/kaggle/working/reward_model_v2/rm_pairs_llava_13k.jsonl")
# nếu bạn đã upload jsonl vào dataset, sửa lại:
JSONL_PATH = DATA_ROOT / "rm_pairs_llava_13k.jsonl"

SFT_CKPT_PATH = DATA_ROOT / "qwen2vl_sft_stage0_best.pt"
RM_CKPT_PATH  = DATA_ROOT / "best_reward_model_v2.pt"

COCO_ROOT = DATA_ROOT   

RL_OUTPUT_DIR = Path("/kaggle/working/qwen2vl_rl_stage1")
RL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("JSONL exists:", JSONL_PATH.exists())
print("SFT_CKPT exists:", SFT_CKPT_PATH.exists())
print("RM_CKPT exists :", RM_CKPT_PATH.exists())
print("COCO_ROOT exists:", COCO_ROOT.exists())

# ==== Device & dtype ====
device = "cuda" if torch.cuda.is_available() else "cpu"
rm_device = "cpu"  
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    torch_dtype = torch.bfloat16
else:
    torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print("Device:", device, "| RM device:", rm_device, "| dtype:", torch_dtype)


2025-12-03 14:38:37.769526: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764772717.791073     369 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764772717.797526     369 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

JSONL exists: True
SFT_CKPT exists: True
RM_CKPT exists : True
COCO_ROOT exists: True
Device: cuda | RM device: cpu | dtype: torch.bfloat16


In [3]:
# =======================
# Cell 3: Load Qwen2-VL + LoRA Stage0 (policy)
# =======================

from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
CACHE_DIR = "/kaggle/working/qwen2vl-cache"  # nơi cache model để lần sau load nhanh

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    cache_dir=CACHE_DIR,
    # Nếu bị thiếu VRAM có thể bật limit pixel:
    # min_pixels=256*28*28,
    # max_pixels=1024*28*28,
)

base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,  # đã set ở Cell 2: bf16 nếu T4 hỗ trợ, không thì fp16
    device_map=None,          # sẽ .to(device) sau
    cache_dir=CACHE_DIR,      # dùng chung cache
    # low_cpu_mem_usage=True, # có thể bật nếu muốn tiết kiệm RAM host
)

# Bật gradient checkpointing để giảm VRAM
base_model.gradient_checkpointing_enable()
base_model.enable_input_require_grads()

# Đưa policy base lên GPU (hoặc CPU nếu không có GPU)
base_model = base_model.to(device)

# LoRA config PHẢI giống SFT Stage0
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

policy_model = get_peft_model(base_model, lora_config)

print("Load SFT LoRA checkpoint từ:", SFT_CKPT_PATH)
state = torch.load(SFT_CKPT_PATH, map_location="cpu")
missing, unexpected = policy_model.load_state_dict(state, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

policy_model = policy_model.to(device)
policy_model.print_trainable_parameters()


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Load SFT LoRA checkpoint từ: /kaggle/input/datasetrm2/qwen2vl_sft_stage0_best.pt
Missing keys: []
Unexpected keys: []
trainable params: 18,464,768 || all params: 2,227,450,368 || trainable%: 0.8290


In [5]:
# =======================
# Cell 5: Load Reward Model v2 (CLIPRewardModelV2) KHÔNG tải CLIP base
# =======================

from transformers import CLIPConfig, CLIPModel, CLIPProcessor

class RewardHead(nn.Module):
    def __init__(self, emb_dim: int):
        super().__init__()
        hidden = 512
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, 1)
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.mlp(features).squeeze(-1)


class CLIPRewardModelV2(nn.Module):
    """
    RM_v2 giống lúc bạn train:
    - dùng img_emb, txt_emb, img_emb*txt_emb, |img_emb-txt_emb|
    - KHÔNG encode length vào model (length xử lý riêng)
    """
    def __init__(self, clip_model: CLIPModel, projection_dim: int):
        super().__init__()
        self.clip_model = clip_model
        feature_dim = 4 * projection_dim
        self.reward_head = RewardHead(feature_dim)

        for p in self.clip_model.parameters():
            p.requires_grad = False

    @torch.no_grad()
    def encode_image(self, pixel_values: torch.Tensor) -> torch.Tensor:
        img_embeds = self.clip_model.get_image_features(pixel_values=pixel_values)
        img_embeds = img_embeds / img_embeds.norm(dim=-1, keepdim=True)
        return img_embeds

    @torch.no_grad()
    def encode_text(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        txt_embeds = self.clip_model.get_text_features(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        txt_embeds = txt_embeds / txt_embeds.norm(dim=-1, keepdim=True)
        return txt_embeds

    def forward(
        self,
        pixel_values: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:
        img_emb = self.encode_image(pixel_values)
        txt_emb = self.encode_text(input_ids, attention_mask)

        mul  = img_emb * txt_emb
        diff = torch.abs(img_emb - txt_emb)

        features = torch.cat([img_emb, txt_emb, mul, diff], dim=-1)
        reward = self.reward_head(features)
        return reward


CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"

# Chỉ tải CONFIG (nhỏ, vài KB), không tải weight 600MB
clip_config = CLIPConfig.from_pretrained(CLIP_MODEL_NAME)

# Khởi tạo CLIP model trống theo config
clip_model = CLIPModel(clip_config).to(rm_device)

# Processor vẫn cần (tokenizer + preprocess), file này nhỏ hơn rất nhiều
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_NAME)

projection_dim = clip_config.projection_dim
reward_model = CLIPRewardModelV2(clip_model, projection_dim).to(rm_device)

print("Load RM ckpt từ:", RM_CKPT_PATH)
rm_state = torch.load(RM_CKPT_PATH, map_location=rm_device)

# RM ckpt của bạn là state_dict của toàn bộ CLIPRewardModelV2 (CLIP + RewardHead)
missing, unexpected = reward_model.load_state_dict(rm_state, strict=False)
print("RM missing keys:", missing)
print("RM unexpected keys:", unexpected)

reward_model.eval()
print("Reward model v2 ready.")


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Load RM ckpt từ: /kaggle/input/datasetrm2/best_reward_model_v2.pt
RM missing keys: []
RM unexpected keys: []
Reward model v2 ready.


In [6]:
# =======================
# Cell 6: RL Dataset từ rm_pairs_llava_13k.jsonl
# =======================

import pandas as pd

assert JSONL_PATH.exists(), f"Không tìm thấy file: {JSONL_PATH}"

records = []
with open(JSONL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        records.append(
            {
                "image_path": obj["image_path"],
                "caption_gold": obj["caption_chosen"],
                "caption_short": obj.get("caption_rejected", ""),
            }
        )

df = pd.DataFrame(records).drop_duplicates(subset=["image_path"]).reset_index(drop=True)
print("Số ảnh unique:", len(df))
df.head()

PROMPT = "Describe the image in a detailed, accurate English caption."

def resolve_image_path(image_path_str: str) -> Path:
    p = COCO_ROOT / image_path_str
    if p.exists():
        return p
    alt = COCO_ROOT / "coco_train2017_subset" / "data13k" / os.path.basename(image_path_str)
    if alt.exists():
        return alt
    raise FileNotFoundError(f"Không tìm thấy ảnh cho: {image_path_str}")

class CocoRLImageDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        img_path = resolve_image_path(row["image_path"])
        image = Image.open(img_path).convert("RGB")
        caption_short = str(row["caption_short"])
        return {
            "image": image,
            "image_path": str(img_path),
            "caption_short": caption_short,
        }

full_rl_dataset = CocoRLImageDataset(df)
print("RL dataset length:", len(full_rl_dataset))
total_len = len(full_rl_dataset)
val_len = int(0.1 * total_len)
test_len = int(0.1 * total_len)
train_len = total_len - val_len - test_len

train_rl, val_rl, test_rl = random_split(
    full_rl_dataset,
    [train_len, val_len, test_len],
    generator=torch.Generator().manual_seed(123),
)

print("RL splits:", len(train_rl), len(val_rl), len(test_rl))


Số ảnh unique: 13000
RL dataset length: 13000
RL splits: 10400 1300 1300


In [7]:
# =======================
# Cell 7: Hàm reward (RM + length bonus)
# =======================

# Length bonus config bạn chọn:
MIN_RATIO = 1.40
TARGET_RATIO = 1.90
MAX_RATIO = 2.50

def compute_length_bonus(
    ratio: float,
    min_ratio: float = MIN_RATIO,
    target_ratio: float = TARGET_RATIO,
    max_ratio: float = MAX_RATIO,
    min_penalty: float = -0.4,
    max_bonus: float = 1.4,
) -> float:
    if ratio <= 1.0:
        return min_penalty

    if ratio < min_ratio:
        t = (ratio - 1.0) / (min_ratio - 1.0 + 1e-8)
        return 0.5 * max_bonus * float(t)

    if ratio <= target_ratio:
        t = (ratio - min_ratio) / (target_ratio - min_ratio + 1e-8)
        return 0.5 * max_bonus + 0.5 * max_bonus * float(t)

    if ratio <= max_ratio:
        t = (ratio - target_ratio) / (max_ratio - target_ratio + 1e-8)
        return max_bonus - 0.3 * max_bonus * float(t)  # 1.0 -> 0.7

    extra = min(ratio - max_ratio, 1.0)
    return max_bonus * (0.7 - 0.4 * extra)  # 0.7 -> 0.3


def scale_rm_to_0_10(raw_score: float, mean: float, std: float) -> float:
    # simple linear scaling: z = (x-mean)/(2*std), map z-> [0,10]
    if std < 1e-6:
        return 5.0
    z = (raw_score - mean) / (2.0 * std)
    s = 5.0 + 4.0 * z
    return float(max(0.0, min(10.0, s)))

@torch.no_grad()
def estimate_rm_stats(num_samples: int = 1000):
    scores = []
    indices = np.random.permutation(len(full_rl_dataset))[:num_samples]
    for idx in tqdm(indices, desc="Estimating RM stats"):
        sample = full_rl_dataset[idx]
        img = sample["image"]
        # dùng caption_gold cho ước lượng
        row = df.iloc[idx]
        cap = str(row["caption_gold"])

        inputs = clip_processor(
            images=[img],
            text=[cap],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=77,
        ).to(rm_device)

        r = reward_model(
            pixel_values=inputs["pixel_values"],
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
        )
        scores.append(r.item())
    arr = np.array(scores)
    return float(arr.mean()), float(arr.std())

RM_MEAN, RM_STD = estimate_rm_stats(num_samples=800)
print("RM_MEAN =", RM_MEAN, "| RM_STD =", RM_STD)

@torch.no_grad()
def compute_reward_for_caption(
    image: Image.Image,
    caption: str,
    caption_short: str,
    alpha_length: float = 0.4,
    rm_gate: float = 4.8,    
):
    # RM score
    rm_inputs = clip_processor(
        images=[image],
        text=[caption],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=77,
    ).to(rm_device)

    raw = reward_model(
        pixel_values=rm_inputs["pixel_values"],
        input_ids=rm_inputs["input_ids"],
        attention_mask=rm_inputs["attention_mask"],
    ).item()
    rm_scaled = scale_rm_to_0_10(raw, RM_MEAN, RM_STD)

    # length ratio
    L_short = max(1, len(str(caption_short).split()))
    L_cap = max(1, len(str(caption).split()))
    ratio = L_cap / L_short
    len_bonus = compute_length_bonus(ratio)

    if rm_scaled < rm_gate:
        effective_len_bonus = 0.0
    else:
        effective_len_bonus = len_bonus

    reward = rm_scaled + alpha_length * effective_len_bonus
    return {
        "rm_raw": raw,
        "rm_scaled": rm_scaled,
        "L_short": L_short,
        "L_cap": L_cap,
        "ratio": ratio,
        "len_bonus": len_bonus,
        "reward": reward,
    }

Estimating RM stats:   0%|          | 0/800 [00:00<?, ?it/s]

RM_MEAN = -42.318043475151065 | RM_STD = 3.4184815424259583


In [8]:
# =======================
# Cell 8: Generate caption + build batch cho RL update
# =======================

@torch.no_grad()
def generate_caption_qwen(model, image: Image.Image, max_new_tokens: int = 80):
    model.eval()
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": PROMPT},
            ],
        }
    ]

    # KHÔNG .to(device) ở đây, để lấy device từ chính model
    inputs = processor.apply_chat_template(
        [messages],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        padding=True,
    )

    # Lấy device của model (policy_model: cuda, ref_model: cpu)
    model_device = next(model.parameters()).device
    inputs = inputs.to(model_device)

    gen_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_p=0.9,
        temperature=0.7,
    )

    # Cắt bỏ phần prompt
    new_tokens_ids = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs["input_ids"], gen_ids)
    ]

    texts = processor.batch_decode(
        new_tokens_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    return texts[0]


def build_sft_inputs_from_batch(images: List[Image.Image], captions: List[str]):
    """
    Dùng lại kiểu SFT:
      user: image + PROMPT
      assistant: caption_gen
    """
    messages_batch = []
    for img, cap in zip(images, captions):
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": PROMPT},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": cap},
                ],
            },
        ]
        messages_batch.append(messages)

    inputs = processor.apply_chat_template(
        messages_batch,
        tokenize=True,
        add_generation_prompt=False,
        return_tensors="pt",
        return_dict=True,
        padding=True,
    )

    input_ids = inputs["input_ids"]
    labels = input_ids.clone()
    inputs["labels"] = labels

    # Ở đây vẫn .to(device) vì chỉ dùng cho policy_model (đang nằm trên `device`)
    return {k: v.to(device) for k, v in inputs.items()}

In [ ]:
# =======================
# Cell 9: RL loop (REINFORCE-style) trên SFT
#         dùng RM_v2 + length bonus, KHÔNG train lại SFT CE_gold
# =======================

from torch.optim import AdamW
import numpy as np
from contextlib import nullcontext

# Hyperparameters RL
RL_STEPS        = 1100     
BATCH_SIZE_RL   = 1       
RL_LR           = 1e-5
MAX_NEW_TOKENS  = 90        
ALPHA_LENGTH    = 0.5
BASELINE_MOMENTUM = 0.9

rl_optimizer = AdamW(policy_model.parameters(), lr=RL_LR)

running_baseline = 0.0
first_baseline = True

def sample_minibatch(dataset, batch_size):
    idxs = np.random.randint(0, len(dataset), size=batch_size)
    return [dataset[i] for i in idxs]


for step in range(1, RL_STEPS + 1):
    if device == "cuda":
        torch.cuda.empty_cache()

    # 1) Lấy batch ảnh
    batch_samples = sample_minibatch(train_rl, BATCH_SIZE_RL)
    images = [s["image"] for s in batch_samples]
    shorts = [s["caption_short"] for s in batch_samples]

    # 2) Generate caption + tính reward (không grad)
    gen_captions = []
    rewards_info = []

    policy_model.eval()
    with torch.no_grad():
        for img, short_cap in zip(images, shorts):
            cap_gen = generate_caption_qwen(
                policy_model,
                img,
                max_new_tokens=MAX_NEW_TOKENS,
            )
            gen_captions.append(cap_gen)

            info = compute_reward_for_caption(
                img,
                cap_gen,
                short_cap,
                alpha_length=ALPHA_LENGTH,
                rm_gate=5.0,      # ngưỡng RM để bật length bonus
            )
            rewards_info.append(info)

    rewards = np.array([ri["reward"] for ri in rewards_info], dtype=np.float32)
    mean_reward = float(rewards.mean())

    # 3) Cập nhật baseline
    if first_baseline:
        running_baseline = mean_reward
        first_baseline = False
    else:
        running_baseline = (
            BASELINE_MOMENTUM * running_baseline
            + (1.0 - BASELINE_MOMENTUM) * mean_reward
        )

    advantages = rewards - running_baseline
    adv_mean = float(advantages.mean())

    # 4) RL update: loss = adv_mean * CE(gen)
    policy_model.train()
    batch_inputs = build_sft_inputs_from_batch(images, gen_captions)

    if device == "cuda":
        amp_ctx = torch.autocast("cuda", dtype=torch_dtype)
    else:
        amp_ctx = nullcontext()

    with amp_ctx:
        outputs = policy_model(**batch_inputs)
        ce_loss = outputs.loss
        loss_rl = adv_mean * ce_loss

    rl_optimizer.zero_grad()
    loss_rl.backward()
    torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 1.0)
    rl_optimizer.step()

    if step % 10 == 0 or step == 1:
        print(
            f"[Step {step}/{RL_STEPS}] "
            f"CE_gen={ce_loss.item():.4f} | "
            f"adv_mean={adv_mean:.4f} | reward_mean={mean_reward:.4f} | baseline={running_baseline:.4f}"
        )
        print("  EXAMPLE CAPTION GEN:")
        print("   short:", shorts[0])
        print("   gen  :", gen_captions[0][:700], "...")
        print("   reward info:", rewards_info[0])
        print("-" * 80)

    # Dọn rác sau khi đã dùng để print xong
    del batch_inputs, outputs
    del batch_samples, images, shorts, gen_captions, rewards_info
    del rewards, advantages

    if device == "cuda":
        torch.cuda.empty_cache()


/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


[Step 1/1100] CE_gen=5.9164 | adv_mean=0.0000 | reward_mean=5.5141 | baseline=5.5141
  EXAMPLE CAPTION GEN:
   short: black and white photo of a bus and some kids.
   gen  : This is a black and white photograph capturing a moment at a train station. The main focus is a train, which is painted in white and has the number 318 268 on its side. The train is stationed on the platform, waiting for its next journey. The platform itself is bustling with activity. Several people can be seen walking around, some carrying luggage, suggesting they might be traveling. In the background, there's a ...
   reward info: {'rm_raw': -41.79823684692383, 'rm_scaled': 5.3041155096355155, 'L_short': 10, 'L_cap': 73, 'ratio': 7.3, 'len_bonus': 0.4199999999999999, 'reward': 5.5141155096355154}
--------------------------------------------------------------------------------
[Step 10/1100] CE_gen=5.3082 | adv_mean=0.5401 | reward_mean=5.8557 | baseline=5.3156
  EXAMPLE CAPTION GEN:
   short: Group of black knife

In [ ]:
# =======================
# Cell 10: Lưu LoRA RL Stage1
# =======================

RL_CKPT_PATH = RL_OUTPUT_DIR / "qwen2vl_rl_stage1_best_V4.pt"
torch.save(policy_model.state_dict(), RL_CKPT_PATH)
print("Saved RL LoRA checkpoint to:", RL_CKPT_PATH)

In [ ]:
# # =======================
# # Cell 4: Tạo reference model (Stage0) để đánh giá sau
# # =======================

# ref_base_model = Qwen2VLForConditionalGeneration.from_pretrained(
#     MODEL_ID,
#     torch_dtype=torch_dtype,
#     device_map=None,
# )
# ref_base_model = ref_base_model.to(device)

# ref_model = get_peft_model(ref_base_model, lora_config)
# ref_state = torch.load(SFT_CKPT_PATH, map_location="cpu")
# ref_model.load_state_dict(ref_state, strict=False)
# ref_model = ref_model.to(device)

# for p in ref_model.parameters():
#     p.requires_grad = False

# ref_model.eval()
# print("Reference model (Stage0) ready (frozen).")
# =======================
# Cell 4: Tạo reference model (Stage0) để đánh giá sau
#  -> ĐỂ TRÊN CPU để tránh OOM cchayj sau
# =======================

REF_DEVICE = "cpu"   # ref_model chỉ để eval, không train, nên để CPU

# Load base model Stage0 trên CPU (mặc định float32)
ref_base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map=None,      # KHÔNG dùng device_map="auto"
)
ref_base_model = ref_base_model.to(REF_DEVICE)

# Quấn LoRA giống hệt policy_model
ref_model = get_peft_model(ref_base_model, lora_config)

# Load LoRA SFT stage0 vào ref_model trên CPU
ref_state = torch.load(SFT_CKPT_PATH, map_location=REF_DEVICE)
load_info = ref_model.load_state_dict(ref_state, strict=False)
print("Ref missing keys:", load_info.missing_keys)
print("Ref unexpected keys:", load_info.unexpected_keys)

ref_model = ref_model.to(REF_DEVICE)

# Đóng băng ref_model (chỉ dùng để so sánh, không train)
for p in ref_model.parameters():
    p.requires_grad = False

ref_model.eval()
print("Reference model (Stage0, CPU) ready. Device:", next(ref_model.parameters()).device)


In [ ]:
# =======================
# Cell 11: Hàm generate + evaluate cho model bất kỳ
# =======================

@torch.no_grad()
def generate_caption_batch(model, samples: List[Dict[str, Any]], max_new_tokens: int = 80):
    model.eval()
    captions = []
    for s in samples:
        img = s["image"]
        cap = generate_caption_qwen(model, img, max_new_tokens=max_new_tokens)
        captions.append(cap)
    return captions


@torch.no_grad()
def eval_model_on_subset(model, dataset, num_samples: int = 100):
    idxs = np.random.permutation(len(dataset))[:num_samples]
    subset = [dataset[i] for i in idxs]

    all_rewards = []
    all_lengths = []
    for s in tqdm(subset, desc="Eval subset"):
        img = s["image"]
        short_cap = s["caption_short"]
        gen_cap = generate_caption_qwen(model, img, max_new_tokens=80)

        info = compute_reward_for_caption(img, gen_cap, short_cap, alpha_length=ALPHA_LENGTH)
        all_rewards.append(info["reward"])
        all_lengths.append(info["L_cap"])

    all_rewards = np.array(all_rewards)
    all_lengths = np.array(all_lengths)

    stats = {
        "reward_mean": float(all_rewards.mean()),
        "reward_std": float(all_rewards.std()),
        "len_mean": float(all_lengths.mean()),
        "len_std": float(all_lengths.std()),
    }
    return stats

In [ ]:
# =======================
# Cell 12: So sánh Stage0 (ref_model) vs Stage1 (policy_model RL)
# =======================

NUM_EVAL = 80  

print("Đánh giá Stage0 (SFT)...")
stage0_stats = eval_model_on_subset(ref_model, val_rl, num_samples=NUM_EVAL)
print("Stage0 stats:", stage0_stats)

print("\nĐánh giá Stage1 (RL)...")
stage1_stats = eval_model_on_subset(policy_model, val_rl, num_samples=NUM_EVAL)
print("Stage1 stats:", stage1_stats)